#  Common

Generated from `src/preprocess/_common.py`.
The first code cell recreates script-like path behavior for notebook execution.


In [1]:
from pathlib import Path
import sys

_NOTEBOOK_BOOTSTRAP_VERBOSE = True

if _NOTEBOOK_BOOTSTRAP_VERBOSE:
    print("[bootstrap] cwd =", Path.cwd().resolve())

if "ipykernel" in sys.modules:
    # Avoid argparse failures from Jupyter kernel launch flags.
    sys.argv = [sys.argv[0]]
    if _NOTEBOOK_BOOTSTRAP_VERBOSE:
        print("[bootstrap] detected ipykernel, trimmed sys.argv to:", sys.argv)

_SOURCE_RELATIVE_PATH = Path("src/preprocess/_common.py")
_repo_root = None
for _candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if _NOTEBOOK_BOOTSTRAP_VERBOSE:
        print("[bootstrap] checking candidate:", _candidate)
    if (_candidate / _SOURCE_RELATIVE_PATH).exists():
        _repo_root = _candidate
        if _NOTEBOOK_BOOTSTRAP_VERBOSE:
            print("[bootstrap] matched repo root:", _repo_root)
        break

if _repo_root is None:
    _repo_root = Path.cwd().resolve()
    if _NOTEBOOK_BOOTSTRAP_VERBOSE:
        print("[bootstrap] no match found, falling back to cwd:", _repo_root)

_source_file = (_repo_root / _SOURCE_RELATIVE_PATH).resolve()
__file__ = str(_source_file)
if _NOTEBOOK_BOOTSTRAP_VERBOSE:
    print("[bootstrap] source relative path =", _SOURCE_RELATIVE_PATH)
    print("[bootstrap] resolved __file__ =", __file__)

for _path in (str(_repo_root), str(_source_file.parent)):
    if _path not in sys.path:
        sys.path.insert(0, _path)
        if _NOTEBOOK_BOOTSTRAP_VERBOSE:
            print("[bootstrap] added to sys.path:", _path)
    elif _NOTEBOOK_BOOTSTRAP_VERBOSE:
        print("[bootstrap] already on sys.path:", _path)


[bootstrap] cwd = C:\Users\Ling Jun\Desktop\PSB\AI-ethics\notebooks\src\preprocess
[bootstrap] detected ipykernel, trimmed sys.argv to: ['c:\\Users\\Ling Jun\\Desktop\\PSB\\AI-ethics\\venv\\lib\\site-packages\\ipykernel_launcher.py']
[bootstrap] checking candidate: C:\Users\Ling Jun\Desktop\PSB\AI-ethics\notebooks\src\preprocess
[bootstrap] checking candidate: C:\Users\Ling Jun\Desktop\PSB\AI-ethics\notebooks\src
[bootstrap] checking candidate: C:\Users\Ling Jun\Desktop\PSB\AI-ethics\notebooks
[bootstrap] checking candidate: C:\Users\Ling Jun\Desktop\PSB\AI-ethics
[bootstrap] matched repo root: C:\Users\Ling Jun\Desktop\PSB\AI-ethics
[bootstrap] source relative path = src\preprocess\_common.py
[bootstrap] resolved __file__ = C:\Users\Ling Jun\Desktop\PSB\AI-ethics\src\preprocess\_common.py
[bootstrap] added to sys.path: C:\Users\Ling Jun\Desktop\PSB\AI-ethics
[bootstrap] added to sys.path: C:\Users\Ling Jun\Desktop\PSB\AI-ethics\src\preprocess


In [2]:
from __future__ import annotations

from pathlib import Path
import csv
import json
from typing import Any, Dict, Iterable, List


def repo_root() -> Path:
    return Path(__file__).resolve().parents[2]


def data_root() -> Path:
    root = repo_root()
    for candidate in (root / "Data", root / "data"):
        if candidate.exists():
            return candidate
    return root / "Data"


RAW_ROOT = data_root() / "raw"
PROCESSED_ROOT = data_root() / "processed"


def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def normalize_for_csv(value: Any) -> Any:
    if value is None:
        return ""
    if isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False)
    return value


def write_jsonl(path: Path, rows: Iterable[Dict[str, Any]]) -> None:
    ensure_dir(path.parent)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "")


def write_csv(path: Path, rows: List[Dict[str, Any]], fieldnames: List[str] | None = None) -> None:
    ensure_dir(path.parent)
    if not rows:
        if fieldnames:
            with path.open("w", encoding="utf-8", newline="") as f:
                writer = csv.DictWriter(f, fieldnames=fieldnames)
                writer.writeheader()
        return
    if fieldnames is None:
        keys: List[str] = []
        seen = set()
        for row in rows:
            for key in row.keys():
                if key not in seen:
                    seen.add(key)
                    keys.append(key)
        fieldnames = keys
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            out = {k: normalize_for_csv(row.get(k)) for k in fieldnames}
            writer.writerow(out)


def read_csv_rows(path: Path, encoding: str = "utf-8", errors: str = "strict") -> List[Dict[str, Any]]:
    with path.open("r", encoding=encoding, errors=errors, newline="") as f:
        reader = csv.DictReader(f)
        return list(reader)


def read_tsv_rows(path: Path, encoding: str = "utf-8", errors: str = "strict") -> List[Dict[str, Any]]:
    with path.open("r", encoding=encoding, errors=errors, newline="") as f:
        reader = csv.DictReader(f, delimiter="	")
        return list(reader)


def read_json_rows(path: Path, encoding: str = "utf-8", errors: str = "strict") -> List[Dict[str, Any]]:
    with path.open("r", encoding=encoding, errors=errors) as f:
        obj = json.load(f)
    if isinstance(obj, list):
        return [item if isinstance(item, dict) else {"value": item} for item in obj]
    if isinstance(obj, dict):
        return [obj]
    return [{"value": obj}]


def read_jsonl_rows(path: Path, encoding: str = "utf-8", errors: str = "strict") -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    with path.open("r", encoding=encoding, errors=errors) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            if isinstance(obj, dict):
                rows.append(obj)
            else:
                rows.append({"value": obj})
    return rows
